# Baseline QA задачи
## Импорт данных

### Описание задачи: Тест, содержащий вопросы с короткими ответами (QA)

Данный датасет представляет собой набор вопросов с **четырьмя вариантами ответов**, где ровно **один вариант является правильным**. Задача модели — по заданному вопросу и списку возможных ответов определить **индекс правильного варианта** (0, 1, 2 или 3).

### Структура данных
Каждая строка содержит следующие поля:
- `id` — уникальный числовой идентификатор вопроса.
- `вопрос` — текст вопроса на русском языке.
- `варианты` — список из четырёх строковых значений, закодированный как JSON-массив (например: `["Тит Ливий", "Тацит", "Плутарх", "Геродот"]`).
- `категория` — тематическая рубрика вопроса (например: *античность*, *аналитическая геометрия для школьников*). Поле может содержать значение `"-"`, если категория не указана.
- `ответ` — целое число от 0 до 3, обозначающее **индекс правильного варианта** в списке `варианты`.

### Пример
| id | вопрос | варианты | категория | ответ |
|----|--------|----------|-----------|-------|
| 0 | Кто из перечисленных не был римским историком? | `["Тит Ливий", "Тацит", "Плутарх", "Геродот"]` | античность | 3 |

> Правильный ответ — **"Геродот"** (индекс 3), так как он был древнегреческим, а не римским историком.

### Цель
Разработать модель (LLM-промпт и т.д.), которая по входным `вопрос` и `варианты` будет предсказывать корректный индекс ответа. Иднексы начинаются от 0. Итоговое решение оценивается по точности (accuracy) на тестовом наборе.

### Особенности
- Вопросы охватывают широкий спектр тем: от истории и литературы до математики и естественных наук.
- Некоторые вопросы требуют **фактических знаний**, другие — **логического вывода** или **понимания формулировок**.
- Категории могут быть использованы как дополнительный признак при построении модели.

In [5]:
import re
import time
import pandas as pd
from tqdm import tqdm
from gigachat import GigaChat
from gigachat.models import Chat, MessagesRole

In [6]:
df_train = pd.read_csv('train.csv')
df_train.head(3)

,id,вопрос,варианты,категория,ответ
0,0.0,Когда был открыт закон Бойля-Мариотта?,"[""1662"", ""1762"", ""1862"", ""1962""]",-,0
1,1.0,Как найти площадь параллелограмма по векторам ...,"[""3"", ""5"", ""6"", ""0""]",аналитическая геометрия для школьников,2
2,2.0,Как реагировать на комплимент?,"[""Отрицать"", ""Смущаться и молчать"", ""Сказать «...",EQ тест,2


Токен

In [7]:
token = open('authGigaChat.txt').read().strip()


Составим функцию, формирующую промпт для запроса

Функция `qa_message_template` формирует **текстовый промпт** для языковой модели (в данном случае — GigaChat) в задаче вопрос-ответ (QA). Её цель — чётко сформулировать задание так, чтобы модель вернула **индекс правильного варианта ответа**, а не сам текст ответа.

Функция объединяет три части в единое текстовое сообщение:
1. **Вопрос** — основной текст задания.
2. **Список вариантов ответа** — передаётся как есть (ожидается строка в формате JSON-массива или простого текста).
3. **Инструкция для модели** — явное указание, что ответ должен быть **номером варианта, начиная с 0**.


In [9]:
def qa_message_template(question = df_train.loc[0,['вопрос']].values[0], 
                        answers = df_train.loc[0,['варианты']].values[0]):
    msg = question
    msg += ' варианты ответа: '
    msg += answers
    msg += ' ответ должен быть номером варианта, считая от 0 '
    return msg
    

Данный фрагмент кода предназначен для **ручной проверки работы модели GigaChat** на конкретном вопросе из обучающего датасета. Он позволяет быстро убедиться, что:
- промпт формируется корректно из `qa_message_template`,
- модель возвращает ожидаемый формат ответа,
- извлечение индекса работает правильно.

In [10]:
idx = 12

question = df_train.loc[idx,['вопрос']].values[0]
answers  = df_train.loc[idx,['варианты']].values[0]

with GigaChat(credentials=token, verify_ssl_certs=False) as giga:
    msg = qa_message_template(question, answers)
    response = giga.chat(msg)
    result = int(re.findall(r'\d+', response.choices[0].message.content)[0])

print('ответ:', result, 'правильный ответ:', df_train.loc[idx,['ответ']].values[0] )    

ответ: 2 правильный ответ: 0




Следующая Выполняет предсказание индекса правильного ответа для каждого вопроса в тестовом датасете
с использованием модели GigaChat.

<!-- Параметры:

* df : pd.DataFrame
    Тестовый датафрейм с колонками: 'id', 'вопрос', 'варианты'.
* token : str
    Авторизационный токен для доступа к GigaChat API.
* max_retries : int, optional
    Максимальное количество попыток при ошибке запроса (по умолчанию 3).
* delay_between_requests : float, optional
    Задержка в секундах между последовательными запросами (по умолчанию 0.5).
* timeout : int, optional
    Таймаут соединения с API в секундах (по умолчанию 60).

Возвращает:

* pd.DataFrame -->
<!-- Датафрейм с колонками: 'id', 'prediction' (значения от 0  или -1 при ошибке). -->

__Функция__ `predict_with_gigachat`

Функция выполняет автоматическую генерацию предсказаний для задачи множественного выбора (multiple-choice QA) с использованием языковой модели **GigaChat** через официальный API.

__Назначение__
Для каждого вопроса из входного датафрейма:
1. Формируется промпт с помощью внешней функции `qa_message_template`.
2. Отправляется запрос в GigaChat.
3. Из текстового ответа модели извлекается **первое целое число** — оно интерпретируется как индекс выбранного варианта ответа (`0`, `1`, `2` или `3`).
4. Результат сохраняется вместе с `id` записи.

Для снижения нагрузки на API и соблюдения лимитов запросов реализована **батчевая обработка**: после обработки заданного числа строк вставляется дополнительная пауза.

__Параметры__
| Параметр | Тип | Описание |
|--------|------|--------|
| `df` | `pd.DataFrame` | Входной датафрейм с колонками: `'id'`, `'вопрос'`, `'варианты'`. |
| `token` | `str` | Авторизационный токен для доступа к GigaChat API. |
| `max_retries` | `int` (опционально) | Максимальное число попыток при ошибке запроса (по умолчанию: `3`). |
| `delay_between_requests` | `float` (опционально) | Задержка в секундах **между отдельными запросами** (по умолчанию: `0.1`). |
| `timeout` | `int` (опционально) | Таймаут соединения с API в секундах (по умолчанию: `60`). |
| `batch_size` | `int` (опционально) | Количество запросов в одном батче после которого делается пауза (по умолчанию: `20`). |
| `delay_between_batches` | `float` (опционально) | Задержка в секундах **после завершения каждого батча** (по умолчанию: `2.0`). |

__Возвращаемое значение__
- `pd.DataFrame` с двумя колонками:
  - `id` — идентификатор вопроса (из исходного датафрейма),
  - `prediction` — предсказанный индекс ответа (`0–3`) или `-1`, если:
    - модель не вернула число,
    - произошла ошибка при запросе (после всех попыток).

__Особенности__
- Используется **контекстный менеджер** `with GigaChat(...)`, что гарантирует корректное закрытие соединения.
- Реализована **обработка исключений**: при сбое (таймаут, сетевая ошибка, rate limit) делается до `max_retries` повторных попыток.
- Поддержка **гибкого управления нагрузкой**:
  - короткая задержка между запросами (`delay_between_requests`),
  - более длительная пауза после каждого батча (`delay_between_batches`).
- Прогресс выполнения отображается с помощью `tqdm`, а после каждого батча выводится информационное сообщение.

> **Требования**:  
> - __Внешняя функция__ `qa_message_template(question, answers)` должна быть определена в глобальной области видимости.  
> - Колонки датафрейма должны называться точно: `'id'`, `'вопрос'`, `'варианты'`.

In [11]:
def predict_with_gigachat(
    df,
    token,
    max_retries=3,
    delay_between_requests=0.0,
    timeout=60,
    batch_size=80,
    delay_between_batches=1.0
):
    """
    Выполняет предсказания с использованием GigaChat с поддержкой:
    - задержки между отдельными запросами,
    - дополнительной паузы после обработки каждого батча (группы строк).
    
    Параметры batch_size и delay_between_batches позволяют гибко управлять нагрузкой на API.
    """
    results = []
    with GigaChat(credentials=token, verify_ssl_certs=False, timeout=timeout) as giga:
        for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Processing QA")):
            
            question, answers, record_id = row['вопрос'], row['варианты'], row['id']
            prediction = -1

            for attempt in range(max_retries):
                try:
                    msg = qa_message_template(question, answers)
                    response = giga.chat(msg)
                    match = re.findall(r'\d+', response.choices[0].message.content.strip())
                    prediction = int(match[0]) if match else -1
                    break  # Успешный запрос — выходим из цикла попыток
                except Exception as e:
                    print(f"\n Ошибка при обработке id={record_id}, попытка {attempt + 1}/{max_retries}: {e}")
                    time.sleep(1 + attempt)
                    prediction = -1

            results.append({'id': record_id, 'prediction': prediction})
            if delay_between_requests>0:
               time.sleep(delay_between_requests)

            # Дополнительная задержка после каждого батча (кроме последнего)
            if (i + 1) % batch_size == 0 and (i + 1) < len(df):
                time.sleep(delay_between_batches)

    return pd.DataFrame(results)

In [13]:
results = predict_with_gigachat(df_train, token = token)

Processing QA: 100%|███████████████████████████████████████████████████████████████████| 87/87 [00:36<00:00,  2.36it/s]


Проверим точность

In [15]:
df_meged = pd.merge(df_train.loc[:,['id','ответ']], results, on='id')
df_meged['is_correct'] = (df_meged['ответ'] ==  df_meged['prediction'])*1
acc = df_meged['is_correct'].mean()
print(f'accuracy { 100*acc:.1f}%')

accuracy 73.6%


## Формирование `submission`

Загрузка данных

In [16]:
df_test = pd.read_csv('test.csv')
df_test.head(3)

,id,вопрос,варианты,категория
0,0.0,Какая страна станет хозяйкой Кубка африканских...,"[""Египет"", ""Марокко"", ""ЮАР"", ""Нигерия""]",-
1,1.0,Какой процесс привёл к появлению торговли?,"[""Неолитическая революция"", ""Промышленная рево...",антропология
2,2.0,"Как называется физическая величина, равная отн...","[""Разрешающая способность"", ""Увеличение"", ""Фок...",оптика


Тест

In [17]:
len(df_test)

786

In [18]:
df_results = predict_with_gigachat(df_test, token = token)

Processing QA: 100%|█████████████████████████████████████████████████████████████████| 786/786 [06:19<00:00,  2.07it/s]


In [19]:
df_results.to_csv('submission.csv', index = False)

## Как будет происходить оценка

In [20]:
submission = pd.read_csv('submission.csv')
answers = pd.read_csv('test_with_answers.csv')

In [24]:
df_meged = pd.merge(answers, submission, on='id')
df_meged['is_correct'] = (df_meged['class'] ==  df_meged['prediction'])*1
acc = df_meged['is_correct'].mean()
print(f'accuracy { 100*acc:.1f}%') 

accuracy 65.8%
